## 1. Environment Setup
Install the required libraries for QLoRA fine-tuning.

In [ ]:
!pip install -q transformers peft bitsandbytes trl accelerate datasets

## 2. Mount Google Drive
Mount Drive to save the trained adapter checkpoints securely.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Load Dataset
Load the DARPA OPTC log-to-technique pairs dataset from Google Drive.

In [ ]:
import json
from datasets import Dataset

# Assume the dataset was uploaded to this path in Drive
dataset_path = '/content/drive/MyDrive/huntgpt/optc_training_pairs.json'

with open(dataset_path, 'r') as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
print(f"Loaded {len(dataset)} training samples.")

## 4. QLoRA Configuration
Configure BitsAndBytes for 4-bit quantization and LoRA for the Mistral attention layers.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # Optimal for A100
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,               # rank - sweet spot
    lora_alpha=32,      # 2x rank
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

## 5. Training
Use SFTTrainer to fine-tune the model on the formatted prompts.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_prompts_func(example):
    # Format matches the expected JSON prompt structure
    log_line = example['log']
    technique_id = example['technique_id']
    technique_name = example.get('technique_name', '')
    hypothesis = example.get('hypothesis', '')
    
    output_json = f'{{"technique_id": "{technique_id}", "technique_name": "{technique_name}", "hypothesis": "{hypothesis}"}}'
    
    text = f"You are a cybersecurity analyst. Analyze this network log and identify the MITRE ATT&CK technique.\nRespond ONLY with valid JSON in this exact format, no other text:\n{{'technique_id': 'T####.###', 'technique_name': '...', 'hypothesis': '...'}}\n\nLog: {log_line}\n\n{output_json}"
    return text

training_args = TrainingArguments(
    output_dir="/content/results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    fp16=False,
    bf16=True, # Optimal for A100
    max_grad_norm=0.3,
    warmup_ratio=0.03,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    formatting_func=formatting_prompts_func,
    max_seq_length=512,
    args=training_args
)

trainer.train()

## 6. Save Adapter to Drive
Persist the trained LoRA adapter and tokenizer to Google Drive for local use.

In [ ]:
output_path = '/content/drive/MyDrive/huntgpt/adapter'
trainer.model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"Model saved to {output_path}")

## 7. Quick Sanity Check
Run 3 test log lines through the model to ensure the JSON outputs look correct.

In [ ]:
import json

# Test logs - substitute with real OPTC test samples
test_logs = [
    "1592398284.456\tC123\t192.168.1.5\t12345\t10.0.0.8\t80\ttcp\thttp",
    "1592398285.456\tC124\t192.168.1.5\t12346\t10.0.0.8\t445\ttcp\tsmb",
    "1592398286.456\tC125\t192.168.1.5\t12347\t8.8.8.8\t53\tudp\tdns\tevil.com"
]

model.eval()
for log_line in test_logs:
    prompt = f"You are a cybersecurity analyst. Analyze this network log and identify the MITRE ATT&CK technique.\nRespond ONLY with valid JSON in this exact format, no other text:\n{{'technique_id': 'T####.###', 'technique_name': '...', 'hypothesis': '...'}}\n\nLog: {log_line}\n\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=False)
        
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the generated JSON portion
    output_text = result[len(prompt):]
    print("LOG:", log_line)
    print("OUTPUT:", output_text.strip())
    print("-" * 40)